# CEFR-SP — download & preprocess

*The on-ramp track: sentence proficiency level (A1–C2)*

**What it is.** Sentences annotated with a CEFR level by two trained annotators. We use the openly-shipped **Wiki-Auto** portion.

**Difficulty of the labeling judgment:** ★☆☆ — easy. Levels are concrete and the annotators usually agree.

**Licence:** CC BY-SA 3.0 (Wiki-Auto portion) — **share-alike**, so anything you redistribute from it inherits the same licence.  
**Cite:** Arase, Uchida & Kajiwara (2022), *EMNLP*. github.com/yukiar/CEFR-SP

---

Every dataset in this course is reshaped into the **same canonical schema**, so one pipeline works for all of them:

```json
[{"id": 1, "text": "...", "label": "..."}]
```

The *raw* data, though, looks different every time. **That difference is the lesson** — half of building a gold standard is getting messy real data into a clean, consistent shape.

> This notebook is **generated** from `scripts/reshape.py`. The reshaping code below is the same code `scripts/prep_datasets.py` runs — not a copy of it. If you want to change how the data is reshaped, edit `reshape.py` and re-run `scripts/_generate_download_notebooks.py`.

## Step 1 — Download the raw data

The corpus lives in a GitHub repository, so we clone it. (`!` runs a shell command from inside the notebook.)

In [ ]:
!git clone --depth 1 https://github.com/yukiar/CEFR-SP

## Step 2 — Look at the raw format

Note the folder path: cloning `CEFR-SP` gives you a `CEFR-SP` folder *inside* `CEFR-SP`. Easy to trip over.

The Wiki-Auto files are **tab-separated text**, one sentence per line:

```
sentence <TAB> label_by_annotator_A <TAB> label_by_annotator_B
```

Labels are digits: `1`=A1, `2`=A2, … `6`=C2.

In [ ]:
RAW_DIR = "CEFR-SP/CEFR-SP/Wiki-Auto"

with open(RAW_DIR + "/CEFR-SP_Wikiauto_dev.txt", encoding="utf-8") as f:
    for _ in range(5):
        print(repr(next(f)))

## Step 3 — Reshape into the canonical schema

Three real decisions happen in the code below, and they are commented where they happen:

1. **Trust only agreement** — keep a sentence only when *both* annotators chose the same level. Every label is then unambiguous, which is what makes this the gentle track. It also means the track is easier than the data really is.
2. **Human-readable labels** — `1` → `A1`, so your prompt can name the levels the way a person would.
3. **Wiki-Auto only** — the repo also ships a `SCoRE/` folder under a *non-commercial* licence. We deliberately never read it. Notice that the code is pointed at the `Wiki-Auto` folder specifically, rather than at the repo root: that is what keeps the two apart.

In [ ]:
def reid(items):
    """Renumber ids sequentially from 1, keeping the current order."""
    renumbered = []
    next_id = 1
    for item in items:
        new_item = dict(item)
        new_item["id"] = next_id
        renumbered.append(new_item)
        next_id = next_id + 1
    return renumbered

CEFR_NUM = {'1': 'A1', '2': 'A2', '3': 'B1', '4': 'B2', '5': 'C1', '6': 'C2'}

def reshape_cefr(wiki_auto_dir):
    """Read the Wiki-Auto TSV files and keep only the sentences both annotators agreed on.

    Each line is:  sentence <TAB> label_by_annotator_A <TAB> label_by_annotator_B
    with labels as digits, 1=A1 ... 6=C2.

    Three real decisions here:
      1. TRUST ONLY AGREEMENT. A sentence is kept only when both annotators chose the
         same level, so every label is unambiguous. That is what makes this the gentle
         on-ramp track - and it also means the track is easier than the data really is.
      2. HUMAN-READABLE LABELS. 1 -> A1, so a prompt can name the levels the way a
         person would.
      3. WIKI-AUTO ONLY. CEFR-SP also ships a SCoRE portion, but it is CC BY-NC-SA
         (non-commercial), so we deliberately do not touch it - see data/SOURCES.md.
    """
    source_dir = Path(wiki_auto_dir)
    rows = []
    # Sorted, so a rebuild reads the files in the same order and ids stay stable.
    for path in sorted(source_dir.glob("*.txt")):
        for line in path.read_text(encoding="utf-8").splitlines():
            parts = line.split("\t")
            if len(parts) < 3:
                continue
            text = parts[0].strip()
            label_a = parts[1].strip()
            label_b = parts[2].strip()
            if text and label_a == label_b and label_a in CEFR_NUM:
                rows.append({"id": 0, "text": text, "label": CEFR_NUM[label_a]})
    return reid(rows)

In [ ]:
rows = reshape_cefr(RAW_DIR)
print("kept", len(rows), "sentences where both annotators agreed")

## Step 4 — Check the label balance

Look at the counts before you trust anything downstream. This corpus is **heavily imbalanced** — B1 and B2 dominate, and A1/C2 are scarce. That is a fact about the data, and it is why the project samples a balanced subset rather than using the pool directly.

In [ ]:
from collections import Counter

print("total items:", len(rows))
print("label counts:", dict(Counter(item["label"] for item in rows)))
rows[:3]        # peek at the first three reshaped items

## A note on what you just built

This is the **pool** — everything usable in the corpus, with its natural label imbalance intact. It is *not* your gold set.

Your gold set comes next, in the project notebook: `sample_pool` draws a *balanced* subset from this pool (equal items per label), which is what makes precision, recall, F1 and the confusion matrix readable. Keeping the two separate also leaves the unsampled items free to serve as few-shot examples without leaking the answers you are testing on.

So: build the pool once, here. Sample from it there.

## Step 5 — Save it

In [ ]:
# Save the pool. Two places you might want it:
#   * this repo, if you cloned it:  "../data/pools/cefr_pool.json"
#   * your Google Drive, so it survives the Colab runtime resetting
import json

OUT_FILE = "cefr_pool.json"

# In Colab, uncomment these two lines to write straight to your Drive:
# from google.colab import drive; drive.mount("/content/drive")
# OUT_FILE = "/content/drive/MyDrive/cefr_pool.json"

with open(OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print("Saved", len(rows), "items to", OUT_FILE)